[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sampriti-sr/MLP_Course/blob/kshitij-variant/docs/toolkit/pandas-basics.ipynb)

# pandas: Series, DataFrames and Selection

<div class="admonition info">
<p class="admonition-title">What you will be able to do after this page</p>
<ul>
<li>Check a table's dtypes and memory and shrink it sensibly.</li>
<li>Choose between .loc and .iloc, and filter with boolean conditions.</li>
<li>Explain index alignment and why arithmetic can produce NaN.</li>
<li>Assign safely and avoid the chained-assignment trap.</li>
</ul>
</div>

**Prerequisites:** <a href="../numpy-arrays/">NumPy: Arrays and Indexing</a>. **Dataset:** Inside Airbnb (London). **Time:** about 40 minutes.

In [1]:
try:
    import mlp_data
except ImportError:  # e.g. a fresh Colab runtime
    %pip install -q "mlp-course @ git+https://github.com/sampriti-sr/MLP_Course@kshitij-variant"
    import mlp_data

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mlp_data import load_airbnb, style

style.apply()
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 12)
pd.set_option("display.float_format", "{:,.2f}".format)

## The situation

pandas is the tool you will use most. It wraps NumPy arrays in a **labelled table**: rows have an index, columns have names and (this matters) *types*. Almost every pandas confusion comes from one of four things: not knowing the dtypes, mixing up label-based and position-based selection, forgetting that arithmetic aligns on the index, or modifying something that is a copy.

Our data is real and a little unwieldy: every Airbnb listing in London (snapshot of September 2026), about 92,000 rows and 90 columns.

In [2]:
airbnb = load_airbnb()
print(airbnb.shape)
airbnb[["id", "neighbourhood_cleansed", "room_type", "accommodates", "bedrooms", "price", "number_of_reviews"]].head()

(91954, 90)


,id,neighbourhood_cleansed,room_type,accommodates,bedrooms,price,number_of_reviews
0,11551,Lambeth,Entire home/apt,5,1.00,$267.50,197
1,13913,Islington,Private room,1,1.00,$127.00,57
2,15400,Kensington and Chelsea,Entire home/apt,2,1.00,$137.50,98
3,24328,Wandsworth,Entire home/apt,2,1.00,$228.00,97
4,25081,Haringey,Entire home/apt,4,2.00,NaN,0


## 1. Know what you loaded: dtypes and memory

Before anything else, ask how big the table is and what *types* its columns are. Wrong dtypes (numbers stored as text, for instance) are the usual root cause of later errors.

In [3]:
dtype_counts = airbnb.dtypes.astype(str).value_counts()
mem = airbnb.memory_usage(deep=True).drop("Index")
print("columns by dtype:", dtype_counts.to_dict())
print(f"total memory: {mem.sum() / 1e6:,.0f} MB")
print("\nfive heaviest columns (MB):")
(mem.sort_values(ascending=False).head(5) / 1e6).round(1).to_frame("MB")

columns by dtype: {'float64': 42, 'str': 28, 'int64': 20}
total memory: 257 MB

five heaviest columns (MB):


,MB
price_quote_raw,46.40
amenities,46.10
description,36.10
host_about,15.30
host_picture_url,11.50


The heaviest columns are free text (descriptions, amenity lists, picture URLs): text stored as Python strings is memory-hungry. Two cheap savings: **select only the columns you need**, and convert columns with few distinct values to the **`category`** dtype.

In [4]:
as_text = airbnb["neighbourhood_cleansed"]
as_category = as_text.astype("category")
print(f"neighbourhood as text: {as_text.memory_usage(deep=True) / 1e6:.1f} MB;  as category: {as_category.memory_usage(deep=True) / 1e6:.2f} MB;  distinct values: {as_text.nunique()}")

keep = ["id", "host_id", "neighbourhood_cleansed", "room_type", "accommodates", "bedrooms", "beds", "price", "number_of_reviews", "review_scores_rating", "availability_365", "minimum_nights"]
small = airbnb[keep].copy()
small["neighbourhood_cleansed"] = small["neighbourhood_cleansed"].astype("category")
small["room_type"] = small["room_type"].astype("category")
print(f"working table: {small.shape[1]} columns, {small.memory_usage(deep=True).sum() / 1e6:.1f} MB (from {mem.sum() / 1e6:,.0f} MB)")
assert small.memory_usage(deep=True).sum() < mem.sum() / 20

neighbourhood as text: 1.7 MB;  as category: 0.09 MB;  distinct values: 33
working table: 12 columns, 8.0 MB (from 257 MB)


A category column stores each distinct value once and a small integer code per row, which is why 33 neighbourhoods shrink to a tiny fraction of their text size. The working table is a small fraction of the original, and everything downstream gets faster.

## 2. Selecting: columns, `.loc` and `.iloc`

- `df["a"]` gives a **Series** (one column); `df[["a", "b"]]` gives a **DataFrame**.
- `.loc[rows, cols]` selects by **label**; the end of a label slice is **included**.
- `.iloc[rows, cols]` selects by **position**; the end is **excluded**, like Python lists.

The difference only shows when the index is not simply 0, 1, 2, ...: so let us give the table a meaningful index.

In [5]:
by_id = small.set_index("id")                         # the listing id becomes the row label
ids = by_id.index[:5].tolist()
print("first five labels:", ids)

print("\n.loc with a label slice (end INCLUDED):")
display_rows = by_id.loc[ids[1]:ids[3], ["room_type", "accommodates"]]
print(display_rows.to_string())

print("\n.iloc with a position slice (end EXCLUDED):")
print(by_id.iloc[1:3, [0, 1]].to_string())

first five labels: [11551, 13913, 15400, 24328, 25081]

.loc with a label slice (end INCLUDED):
             room_type  accommodates
id                                  
13913     Private room             1
15400  Entire home/apt             2
24328  Entire home/apt             2

.iloc with a position slice (end EXCLUDED):
       host_id  neighbourhood_cleansed
id                                    
13913    54730               Islington
15400    60302  Kensington and Chelsea


`.loc[ids[1]:ids[3]]` returned three rows (both ends), `.iloc[1:3]` returned two. **If you remember one thing: `.loc` = labels, inclusive; `.iloc` = positions, exclusive.** Mixing them up is silent when the index is the default 0..n−1, which is why bugs appear only after you sort or filter.

## 3. Filtering rows with boolean conditions

Build a True/False Series and use it to select rows. Combine conditions with `&`, `|`, `~`, with **parentheses around each condition**.

In [6]:
is_home = small["room_type"] == "Entire home/apt"
family_size = (small["accommodates"] >= 4) & (small["bedrooms"] >= 2)
print(f"entire homes: {is_home.sum():,};  of those sleeping 4+ with 2+ bedrooms: {(is_home & family_size).sum():,}")

# three equivalent ways to ask for 'entire homes in Westminster or Camden'
a = small[(small["room_type"] == "Entire home/apt") & small["neighbourhood_cleansed"].isin(["Westminster", "Camden"])]
b = small.query("room_type == 'Entire home/apt' and neighbourhood_cleansed in ['Westminster', 'Camden']")
c = small.loc[is_home & small["neighbourhood_cleansed"].isin(["Westminster", "Camden"])]
assert len(a) == len(b) == len(c)
print(f"entire homes in Westminster or Camden: {len(a):,}")

entire homes: 61,074;  of those sleeping 4+ with 2+ bedrooms: 30,153
entire homes in Westminster or Camden: 13,117


`.isin([...])` replaces a chain of `or`s; `.between(lo, hi)` handles ranges; `.query("...")` reads like English and is convenient for long conditions. All three styles above are equivalent, so choose the clearest.

## 4. Counting and ranking

`value_counts` is the workhorse. Combined with a little arithmetic it answers real questions. Airbnb listings are supposed to be people renting out their home, but who actually owns them?

In [7]:
per_host = small["host_id"].value_counts()               # listings per host
pros = per_host[per_host >= 10]
print(f"{len(per_host):,} hosts in total; {len(pros):,} of them ({len(pros) / len(per_host):.1%}) have 10 or more listings")
print(f"those {len(pros):,} hosts own {pros.sum() / len(small):.1%} of all listings; the biggest host has {per_host.iloc[0]} listings")

small["neighbourhood_cleansed"].value_counts().head(5).to_frame("listings")

51,348 hosts in total; 890 of them (1.7%) have 10 or more listings
those 890 hosts own 24.6% of all listings; the biggest host has 501 listings


,listings
neighbourhood_cleansed,
Westminster,10825
Tower Hamlets,6723
Kensington and Chelsea,6271
Camden,6178
Hackney,5740


A tiny share of hosts own about a quarter of all the listings: the market is much more commercial than the "spare room" image suggests. (We will meet this again: modelling price as if every listing were independent ignores that one operator may price hundreds of them identically.)

## 5. Alignment: pandas matches on labels, not positions

When you do arithmetic between two Series, pandas lines them up by **index label**. Labels present in only one side give `NaN`. This is powerful and a classic source of surprise.

In [8]:
west = by_id[by_id["neighbourhood_cleansed"] == "Westminster"]["accommodates"].head(4)
cam = by_id[by_id["neighbourhood_cleansed"] == "Camden"]["accommodates"].head(4)
print("Westminster ids:", west.index.tolist(), "| Camden ids:", cam.index.tolist())
print("\nwest + cam (aligned on id; no ids in common):")
print((west + cam).to_string())

Westminster ids: [17402, 39387, 2912510, 45168] | Camden ids: [2889261, 2916820, 45163, 47192]

west + cam (aligned on id; no ids in common):
id
17402     NaN
39387     NaN
45163     NaN
45168     NaN
47192     NaN
2889261   NaN
2912510   NaN
2916820   NaN


Every result is `NaN` because no listing id appears in both. The `+` did not pair "the first Westminster listing with the first Camden listing"; it paired *equal ids*. To combine by position instead, strip the labels (`west.to_numpy() + cam.to_numpy()`) or `reset_index(drop=True)` first.

## 6. Copies, views and the chained-assignment trap

Assigning into the result of a filter usually does **nothing** to the original table:

In [9]:
work = small.copy()
before = work["accommodates"].sum()

import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    work[work["room_type"] == "Shared room"]["accommodates"] = 99      # chained: filter first, then assign into the result

print("chained assignment changed the table:", work["accommodates"].sum() != before)

work.loc[work["room_type"] == "Shared room", "accommodates"] = 99      # one .loc call selects rows and columns together
print("a single .loc call changed it:        ", work["accommodates"].sum() != before)
assert work["accommodates"].sum() != before

chained assignment changed the table: False
a single .loc call changed it:         True


The first line filtered (producing a temporary copy) and then assigned into the *copy*, which was thrown away. Older pandas warned about this with a `SettingWithCopyWarning`; **pandas 3's copy-on-write makes the rule absolute**: a filtered result is always its own object, so chained assignment never reaches the original. The fix is a single `.loc[rows, column] = value`, which is also the only form that is guaranteed to work in every version.

## Pitfalls at a glance

| Surprise | Cause | Fix |
|---|---|---|
| `.loc[2:4]` returns one more row than expected | label slices include the end | use `.iloc[2:4]` for positions |
| `ValueError` on `a > 1 & a < 5` | operator precedence | `(a > 1) & (a < 5)` |
| Arithmetic gives lots of `NaN` | labels do not match | align deliberately, or `to_numpy()` |
| Assignment "did nothing" | chained assignment on a copy | `df.loc[mask, "col"] = value` |
| Table uses far too much memory | text columns and wide dtypes | select columns; `category`; smaller dtypes |

## Check yourself

<details class="question">
<summary>1. After <code>df = df.sort_values("price")</code>, <code>df.iloc[0]</code> and <code>df.loc[0]</code> may differ. Why?</summary>

Sorting reorders the rows but keeps each row's original index label. `.iloc[0]` is the **first row by position** (now the cheapest), while `.loc[0]` is the **row labelled 0** (wherever it ended up). They agree only while the index is still in its original order.

</details>

<details class="question">
<summary>2. Why does <code>df[df.x > 5]["y"] = 0</code> leave <code>df</code> unchanged?</summary>

The first part, `df[df.x > 5]`, builds a new filtered table; the `["y"] = 0` then assigns into that new object, which is discarded. Use `df.loc[df.x > 5, "y"] = 0`, which selects and assigns on the original in one step.

</details>

<details class="question">
<summary>3. You add two Series of equal length and get all <code>NaN</code>. What is the likely cause?</summary>

Their **index labels do not match**. pandas aligns by label, not position, so with no shared labels every result is missing. Check `a.index` and `b.index`, or drop to arrays with `to_numpy()` if you really want positional pairing.

</details>

## Exercises

<details class="question">
<summary>Among entire homes, which neighbourhood has the most listings, and what share of all entire homes is that? Which neighbourhood has the highest <em>average number of bedrooms</em>? (Hint: <code>groupby</code> arrives on the third pandas page; try <code>value_counts</code> for the first part.)</summary>

```python
homes = small[small["room_type"] == "Entire home/apt"]
counts = homes["neighbourhood_cleansed"].value_counts()
print(counts.index[0], f"{counts.iloc[0]:,} listings = {counts.iloc[0] / len(homes):.1%} of entire homes")
```

Westminster comes first with about 8,800 listings, roughly one in seven entire homes. For the second part you need `homes.groupby("neighbourhood_cleansed")["bedrooms"].mean()`, covered next.

</details>

<details class="question">
<summary>Create a boolean column <code>professional</code> that is True for listings whose host has 10 or more listings, then report how many listings it flags.</summary>

```python
small["professional"] = small["host_id"].map(per_host) >= 10
print(small["professional"].sum(), f"({small['professional'].mean():.1%})")
```

`per_host` is a Series indexed by `host_id`, so `.map(per_host)` looks each listing's host up in it and returns that host's listing count. The result flags about a quarter of all listings, matching the 24.6% computed earlier.

</details>

## Recap

- Check **dtypes and memory** first. Select needed columns and use `category` for low-cardinality text.
- **`.loc` = labels (end included); `.iloc` = positions (end excluded).**
- Filter with boolean masks (`&`, `|`, `~`, parenthesised), `isin`, `between` or `query`.
- Arithmetic **aligns on index labels**: no shared labels means `NaN`.
- Assign with a single `.loc[mask, column] = value`; chained assignment writes into a discarded copy.

**Next:** *Cleaning and types*, where these tools meet genuinely messy columns.